# A5. Inventory Reorder Decision Notebook

> **Companion module**: [A5 Inventory & Supply Chain](../paths/a-operators/a5-inventory.md)
>
> **Function**: Predict reorder quantity and timing based on sales trends + Safety stock calculation
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a5-inventory-reorder.ipynb)

---

## 1. Install Dependencies

In [ ]:
!pip install -q pandas numpy plotly

## 2. Input Inventory Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# === Replace with your actual data ===
skus = [
    {
        'sku': 'SKU-001', 'name': 'Wireless Earbuds',
        'current_inventory': 850,
        'daily_sales_avg': 15, 'daily_sales_std': 5,
        'lead_time_days': 45,  # Days from order to warehouse arrival
        'moq': 500,            # Minimum order quantity
        'unit_cost': 8.50,
        'fba_monthly_storage': 0.15  # Monthly storage fee per unit
    },
    {
        'sku': 'SKU-002', 'name': 'Phone Stand',
        'current_inventory': 1200,
        'daily_sales_avg': 27, 'daily_sales_std': 8,
        'lead_time_days': 35,
        'moq': 1000,
        'unit_cost': 3.20,
        'fba_monthly_storage': 0.10
    },
    {
        'sku': 'SKU-003', 'name': 'LED Desk Lamp',
        'current_inventory': 180,
        'daily_sales_avg': 10, 'daily_sales_std': 4,
        'lead_time_days': 50,
        'moq': 300,
        'unit_cost': 7.00,
        'fba_monthly_storage': 0.25
    }
]

SERVICE_LEVEL = 0.95  # 95% service level (probability of no stockout)
Z_SCORE = 1.65        # Z-value corresponding to 95% service level

print(f'Loaded {len(skus)} SKUs')
for s in skus:
    days = s['current_inventory'] / s['daily_sales_avg']
    status = '🔴' if days < 14 else ('🟡' if days < 30 else '🟢')
    print(f"{status} {s['name']}: {s['current_inventory']} units, {days:.0f} days of supply")

## 3. Safety Stock + Reorder Point Calculation

In [ ]:
results = []
for s in skus:
    # Safety stock = Z × σ_daily × √(lead_time)
    safety_stock = Z_SCORE * s['daily_sales_std'] * np.sqrt(s['lead_time_days'])
    
    # Reorder point = (daily avg sales × lead time) + safety stock
    reorder_point = (s['daily_sales_avg'] * s['lead_time_days']) + safety_stock
    
    # Days of supply
    days_of_supply = s['current_inventory'] / s['daily_sales_avg']
    
    # Need to reorder?
    needs_reorder = s['current_inventory'] <= reorder_point
    
    # Suggested reorder quantity (cover 60 days + safety stock)
    target_inventory = s['daily_sales_avg'] * 60 + safety_stock
    reorder_qty = max(target_inventory - s['current_inventory'] + s['daily_sales_avg'] * s['lead_time_days'], s['moq'])
    reorder_qty = int(np.ceil(reorder_qty / s['moq']) * s['moq'])  # Round up to MOQ
    
    # Estimated stockout date
    stockout_date = datetime.now() + timedelta(days=days_of_supply)
    
    # Latest order date
    latest_order_date = stockout_date - timedelta(days=s['lead_time_days'])
    
    # Reorder cost
    reorder_cost = reorder_qty * s['unit_cost']
    
    results.append({
        'SKU': s['sku'], 'Product': s['name'],
        'Current': s['current_inventory'],
        'Daily Avg': s['daily_sales_avg'],
        'Days Supply': round(days_of_supply, 1),
        'Safety Stock': round(safety_stock),
        'Reorder Point': round(reorder_point),
        'Needs Reorder': needs_reorder,
        'Reorder Qty': reorder_qty,
        'Reorder Cost': reorder_cost,
        'Stockout Date': stockout_date.strftime('%Y-%m-%d'),
        'Latest Order': latest_order_date.strftime('%Y-%m-%d'),
        'Lead Time': s['lead_time_days']
    })

df = pd.DataFrame(results)
print('=== Reorder Decision Analysis ===')
for _, r in df.iterrows():
    emoji = '🔴 Reorder Now' if r['Needs Reorder'] else '🟢 No Reorder Needed'
    print(f"\n{emoji}: {r['Product']} ({r['SKU']})")
    print(f"  Inventory: {r['Current']} units | Supply: {r['Days Supply']} days")
    print(f"  Safety Stock: {r['Safety Stock']} units | Reorder Point: {r['Reorder Point']} units")
    if r['Needs Reorder']:
        print(f"  ⚡ Suggested Reorder: {r['Reorder Qty']} units (${r['Reorder Cost']:,.0f})")
        print(f"  ⏰ Latest Order Date: {r['Latest Order']} | Est. Stockout: {r['Stockout Date']}")
    else:
        print(f"  Est. Stockout: {r['Stockout Date']}")

## 4. Inventory Depletion Forecast Visualization

In [ ]:
fig = go.Figure()
colors = ['blue', 'green', 'orange']

for i, s in enumerate(skus):
    days_range = range(0, 90)
    inventory_projection = [max(s['current_inventory'] - s['daily_sales_avg'] * d, 0) for d in days_range]
    dates = [datetime.now() + timedelta(days=d) for d in days_range]
    
    fig.add_trace(go.Scatter(
        x=dates, y=inventory_projection,
        name=s['name'], line=dict(color=colors[i])
    ))
    
    # Reorder point line
    rp = results[i]['Reorder Point']
    fig.add_hline(y=rp, line_dash='dash', line_color=colors[i],
                  annotation_text=f"{s['name']} Reorder Point")

fig.add_hline(y=0, line_color='red', line_width=2, annotation_text='STOCKOUT')
fig.update_layout(title='90-Day Inventory Projection', xaxis_title='Date', yaxis_title='Units')
fig.show()

## 5. Export

In [ ]:
df.to_csv('reorder_decisions.csv', index=False)
print('Exported to reorder_decisions.csv')

# Urgent reorder summary
urgent = df[df['Needs Reorder']]
if len(urgent) > 0:
    total_cost = urgent['Reorder Cost'].sum()
    print(f'\n⚠️ {len(urgent)} SKUs need immediate reorder')
    print(f'Total reorder cost: ${total_cost:,.0f}')
else:
    print('\n✅ All SKUs have sufficient inventory')